## Creating an a summary agent for control performance.
### Approach to be used
- Create an endpoint for a table that will store the control summary
- An agent that will retrive control exceptions and relevant context using RAG.
- An agent should review those exceptions and record an insightful summary.
- An agent should review that summary and determine if it sufficient to be recorded.

Firstly, create and endpoint to be used to store agent feedback

Import all necessary libraries

In [1]:
#import os
from dotenv import load_dotenv
from agents import Agent, Runner,trace
import requests
#import json
load_dotenv(override=True)

True

Retrieve control exceptions

In [2]:
base_URL = 'https://controlweb-supabase.azurewebsites.net'
end_point= '/data/exception'
directory = base_URL+end_point

response = requests.get(directory)

if response.status_code == 200:
    results = response.json()


Create a dictionary that has a key and value, the value should be exceptions

In [3]:
control_exceptions = {"exceptions":results}

In [21]:
print(control_exceptions)

{'exceptions': [{'phone': '+12345678909', 'account_number': 'ACC010', 'registration_date': '2019-11-20', 'name': 'Charles Hernandez', 'status': 'Suspended', 'usage_amount': 90.0, 'email': 'charles.hernandez@example.com', 'timestamp': '2023-09-01 08:25:00', 'user_id': 'USR010', 'detection_time': '2026-04-02T16:40:09.087326'}, {'phone': '+12345678941', 'account_number': 'ACC042', 'registration_date': '2021-09-12', 'name': 'Isaac Harris', 'status': 'Suspended', 'usage_amount': 200.0, 'email': 'isaac.harris@example.com', 'timestamp': '2023-09-01 17:50:00', 'user_id': 'USR042', 'detection_time': '2026-04-02T16:40:09.087326'}]}


Extract the control logic

In [4]:
base_URL = 'https://controlweb-supabase.azurewebsites.net'
end_point= '/data/logic'
directory = base_URL+end_point

response = requests.get(directory)

if response.status_code == 200:
    results = response.json()

In [5]:
control_info ={"control_logic":results}

In [25]:
print(control_info)

{'control_logic': [{'control_logic': "SELECT *\n                    FROM raw_control_datalake.dev.raw_synthetic_data\n                    WHERE status = 'Suspended';\n                    ", 'created_timestamp': '2026-04-08T10:20:42.380718', 'control_logic_description': 'Check suspended customers', 'reference_number': 1, 'control_logic_status': 'ACTIVE'}]}


extract data dictionary

In [6]:
base_URL = 'https://controlweb-supabase.azurewebsites.net'
end_point= '/data/dictionary'
directory = base_URL+end_point

response = requests.get(directory)

if response.status_code == 200:
    results = response.json()

In [27]:
print(results)

[{'field_name': 'User ID', 'description': 'Unique identifier for each user', 'normal_pattern': 'USR001, USR002, etc.', 'modified_timestamp': None, 'data_type': 'string', 'anomaly_indicators': 'None', 'ingestion_timestamp': '2026-04-02T16:34:03.317002'}, {'field_name': 'Usage Amount', 'description': 'Total monthly usage in dollars', 'normal_pattern': '10 - 100', 'modified_timestamp': None, 'data_type': 'number', 'anomaly_indicators': 'Outliers or extremely high values', 'ingestion_timestamp': '2026-04-02T16:34:03.317002'}, {'field_name': 'Transaction Timestamp', 'description': 'Timestamp of the transaction', 'normal_pattern': 'YYYY-MM-DD HH:MM:SS', 'modified_timestamp': None, 'data_type': 'string', 'anomaly_indicators': 'Uncommon hours or dates', 'ingestion_timestamp': '2026-04-02T16:34:03.317002'}, {'field_name': 'Status', 'description': 'Current status of the account', 'normal_pattern': 'Active, Suspended', 'modified_timestamp': None, 'data_type': 'string', 'anomaly_indicators': 'Mark

In [7]:
dictionary = {"data_dictionary":results}

In [8]:
system_prompt = """You are a Fraud Analyst assistant specializing in the review of automated control exceptions.
Your role and context:
You will be provided with three inputs for each review session:

Data dictionary — definitions and descriptions of all relevant data fields
Control information — the control's name, description, and the logic used to generate exceptions
Exception list — the records flagged by the control

Your primary task:
Upon receiving these inputs, produce a structured summary that covers:

A plain-language explanation of what the control is designed to detect and why it matters from a fraud risk perspective
A description of the exception population (volume, key patterns, notable characteristics)
An interpretation of the exception data in the context of the control logic, highlighting anything unusual, unexpected, or high-priority
Any data quality observations or limitations that may affect the reliability of the exceptions

Guidelines:

Ground all observations strictly in the data provided — do not speculate beyond what the inputs support
Use the data dictionary to ensure accurate interpretation of field values
Flag ambiguities where the control logic or data is unclear
Keep the summary concise and actionable, prioritising information that would help a reviewer triage or escalate exceptions"""

In [9]:
fraud_analyst = Agent(name="Fraud analyst",
                      instructions=system_prompt,
                      model="gpt-4o-mini" 
                      )

In [10]:
message = f"Provide a summary for the following control, use the following data:\n\n {control_info} \n\n, {control_exceptions} \n\n, {dictionary}"

In [11]:
print(message)

Provide a summary for the following control, use the following data:

 {'control_logic': [{'control_logic': "SELECT *\n                    FROM raw_control_datalake.dev.raw_synthetic_data\n                    WHERE status = 'Suspended';\n                    ", 'created_timestamp': '2026-04-08T10:20:42.380718', 'control_logic_description': 'Check suspended customers', 'reference_number': 1, 'control_logic_status': 'ACTIVE'}]} 

, {'exceptions': [{'phone': '+12345678909', 'account_number': 'ACC010', 'registration_date': '2019-11-20', 'name': 'Charles Hernandez', 'status': 'Suspended', 'usage_amount': 90.0, 'email': 'charles.hernandez@example.com', 'timestamp': '2023-09-01 08:25:00', 'user_id': 'USR010', 'detection_time': '2026-04-02T16:40:09.087326'}, {'phone': '+12345678941', 'account_number': 'ACC042', 'registration_date': '2021-09-12', 'name': 'Isaac Harris', 'status': 'Suspended', 'usage_amount': 200.0, 'email': 'isaac.harris@example.com', 'timestamp': '2023-09-01 17:50:00', 'user_id

In [14]:

result = await Runner.run(fraud_analyst,message)

In [15]:
print(result.final_output)

### Control Summary

**Control Overview:**
The control is designed to identify and flag accounts with a status of 'Suspended'. This is significant from a fraud risk perspective as suspended accounts may indicate unusual or suspicious activity. The detection of these accounts is crucial for mitigating risks associated with fraudulent transactions or misuse of services.

**Exception Population:**
- **Total Exceptions:** 2
- **Key Patterns:**
  - Both flagged accounts have a status of 'Suspended'.
  - The accounts belong to different users and have varying registration dates and usage amounts.
- **Notable Characteristics:**
  - Account 'ACC010' registered on 2019-11-20, with a usage amount of $90.00.
  - Account 'ACC042' registered more recently on 2021-09-12, with a significantly higher usage amount of $200.00.

**Interpretation of Exception Data:**
- Both accounts were detected on 2026-04-02, which aligns with the control's logic to identify suspended accounts. 
- The detection timestam